In [ ]:
# Cellule 1 - Importer les bibliothèques
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats import ttest_ind

In [ ]:
# Cellule 2 - Charger les logs
log_path = Path("../logs/ab_predictions.jsonl")

df = pd.read_json(log_path, lines=True)
print(df.shape)
df.head()

In [ ]:
# Cellule 3 - Vérifier les colonnes

df.columns

In [ ]:
# Cellule 4 - Compter le nombre de requêtes par variante

df["variant"].value_counts()

In [ ]:
# Cellule 5 - Comparer les prédictions moyennes

df.groupby("variant")["prediction"].agg(["count", "mean", "std", "min", "max"])

In [ ]:
# Cellule 6 - Visualiser les distributions

for variant in ["A", "B"]:
    subset = df[df["variant"] == variant]
    plt.hist(subset["prediction"], alpha=0.5, bins=30, label=variant)

plt.legend()
plt.title("Distribution des prédictions par variante")
plt.xlabel("prediction")
plt.ylabel("nombre de requêtes")
plt.show()

In [ ]:
# Cellule 7 - Comparer la latence

df.groupby("variant")["latency_ms"].agg(["count", "mean", "std", "max"])

In [ ]:
# Cellule 8 - Faire un T-test sur les prédictions

group_a = df[df["variant"] == "A"]["prediction"]
group_b = df[df["variant"] == "B"]["prediction"]

stat, pvalue = ttest_ind(group_a, group_b, equal_var=False)

print("statistique =", stat)
print("p-value =", pvalue)

In [ ]:
# Cellule 9 : Ajouter une ground truth simulée

import numpy as np

# Simulation pédagogique d'une vérité terrain
# On suppose ici que la valeur réelle est proche de la prédiction,
# avec un bruit aléatoire ajouté.
df["actual_value"] = df["prediction"] + np.random.normal(
    loc=0,
    scale=0.3,
    size=len(df),
)

df.head()

In [ ]:
# Cellule 10 : Calculer l’erreur absolue

df["absolute_error"] = (df["actual_value"] - df["prediction"]).abs()

df.groupby("variant")["absolute_error"].agg(["count", "mean", "std"])

In [ ]:
# Cellule 11 : T-test sur l’erreur

from scipy.stats import ttest_ind

err_a = df[df["variant"] == "A"]["absolute_error"]
err_b = df[df["variant"] == "B"]["absolute_error"]